In [1]:
import pandas as pd
import numpy as np
from sklearn.decomposition import FactorAnalysis
from sklearn.preprocessing import StandardScaler

# 1. Load the dataset
df_survey = pd.read_csv('student_survey_fa.csv')

# 2. Standardize
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_survey)

# 3. Fit Factor Analysis to get the loadings
fa = FactorAnalysis(n_components=2, random_state=42)
fa.fit(X_scaled)
loadings = pd.DataFrame(fa.components_.T, columns=['Factor 1', 'Factor 2'], index=df_survey.columns)

# Define the items for each construct based on our knowledge of the data
tech_items = ['Q1_Coding_Skill', 'Q2_Software_Adaptability', 'Q3_Troubleshooting']
mot_items = ['Q4_Study_Hours', 'Q5_Assignment_Punctuality', 'Q6_Class_Attendance']

# Helper function to calculate Cronbach's Alpha
def cronbach_alpha(df):
    df_corr = df.corr()
    N = df.shape[1]
    rs = np.array([])
    for i, col in enumerate(df_corr.columns):
        sum_ = df_corr[df_corr.columns].values[i]
        sum_ = np.delete(sum_, i)
        rs = np.append(rs, sum_)
    mean_r = np.mean(rs)
    alpha = (N * mean_r) / (1 + (N - 1) * mean_r)
    return alpha

# Helper function to calculate AVE and CR
def calculate_ave_cr(loadings_series):
    # Squared loadings
    sq_loadings = loadings_series ** 2
    # Error variance
    e_var = 1 - sq_loadings
    
    # Average Variance Extracted (AVE)
    ave = sq_loadings.sum() / len(loadings_series)
    
    # Composite Reliability (CR)
    cr = (loadings_series.sum() ** 2) / ((loadings_series.sum() ** 2) + e_var.sum())
    
    return ave, cr

# Calculate metrics for Tech-Savviness (Factor 1)
alpha_tech = cronbach_alpha(df_survey[tech_items])
tech_loadings = loadings.loc[tech_items, 'Factor 1']
ave_tech, cr_tech = calculate_ave_cr(tech_loadings)

# Calculate metrics for Academic Motivation (Factor 2)
alpha_mot = cronbach_alpha(df_survey[mot_items])
mot_loadings = loadings.loc[mot_items, 'Factor 2']
ave_mot, cr_mot = calculate_ave_cr(mot_loadings)

print("--- MEASUREMENT MODEL ADEQUACY (OUTER MODEL) ---")
print("\nConstruct 1: Tech-Savviness")
print(f"Cronbach's Alpha       : {alpha_tech:.4f} (Target: > 0.7)")
print(f"Composite Reliability  : {cr_tech:.4f} (Target: > 0.7)")
print(f"Average Var Extracted  : {ave_tech:.4f} (Target: > 0.5)")

print("\nConstruct 2: Academic Motivation")
print(f"Cronbach's Alpha       : {alpha_mot:.4f} (Target: > 0.7)")
print(f"Composite Reliability  : {cr_mot:.4f} (Target: > 0.7)")
print(f"Average Var Extracted  : {ave_mot:.4f} (Target: > 0.5)")

--- MEASUREMENT MODEL ADEQUACY (OUTER MODEL) ---

Construct 1: Tech-Savviness
Cronbach's Alpha       : 0.9111 (Target: > 0.7)
Composite Reliability  : 0.8967 (Target: > 0.7)
Average Var Extracted  : 0.7433 (Target: > 0.5)

Construct 2: Academic Motivation
Cronbach's Alpha       : 0.9038 (Target: > 0.7)
Composite Reliability  : 0.8866 (Target: > 0.7)
Average Var Extracted  : 0.7227 (Target: > 0.5)
